# Class decorators

In [1]:
from functools import wraps
from types import FunctionType

In [2]:
def wrapper(function):
    @wraps(function)
    def wrapped(self, *args, **kwargs):
        if function.__name__ == '__init__':
            print(f'{self.__class__.__name__} is being initialized')
        else:
            print(f'Decorated: {function.__name__} of {self.__class__.__name__}')
        function(self, *args, **kwargs)
        print(f'finished executing {function.__name__}')
    return wrapped

class TestMeta(type):
    def __new__(meta, classname, bases, class_dict):
        new_class_dict = {}
        for attribute_name, attribute in class_dict.items():
            if isinstance(attribute, FunctionType):
                attribute = wrapper(attribute)
            new_class_dict[attribute_name] = attribute
        return type.__new__(meta, classname, bases, new_class_dict)

class TestClass(metaclass=TestMeta):

    def __init__(self):
        print("This is my __init__ of TestClass")

    def method(self):
        print("This is my method of TestClass")
        
x = TestClass()
x.method()

TestClass is being initialized
This is my __init__ of TestClass
finished executing __init__
Decorated: method of TestClass
This is my method of TestClass
finished executing method


## Optional wrapper

In [3]:
def wrapper1(function):
    @wraps(function)
    def wrapped(self, *args, **kwargs):
        if function.__name__ == '__init__':
            print(f'{self.__class__.__name__} is being initialized 11')
        else:
            print(f'Decorated: {function.__name__} of {self.__class__.__name__} 11')
        function(self, *args, **kwargs)
        print(f'finished executing {function.__name__} 11')
    return wrapped

def meta_decor(decorator):
    class TestMeta(type):
        def __new__(meta, classname, bases, class_dict):
            new_class_dict = {}
            for attribute_name, attribute in class_dict.items():
                if isinstance(attribute, FunctionType):
                    attribute = decorator(attribute)
                new_class_dict[attribute_name] = attribute
            return type.__new__(meta, classname, bases, new_class_dict)
    return TestMeta

class TestClass(metaclass=meta_decor(wrapper1)):

    def __init__(self):
        print("This is my __init__ of TestClass")

    def method(self):
        print("This is my method of TestClass")
        
x = TestClass()
x.method()

TestClass is being initialized 11
This is my __init__ of TestClass
finished executing __init__ 11
Decorated: method of TestClass 11
This is my method of TestClass
finished executing method 11


In [4]:
class DecorateMethods(type):
    """ Decorate all methods of the class with the decorator provided """

    def __new__(cls, name, bases, attrs, **kwargs):
        try:
            decorator = kwargs['decorator']
        except KeyError:
            raise ValueError('Please provide the "decorator" argument, eg. '
                             'MyClass(..., metaclass=DecorateMethods, decorator=my_decorator)')

        exclude = kwargs.get('exclude', [])

        for attr_name, attr_value in attrs.items():

            if isinstance(attr_value, FunctionType) and \
                    attr_name not in exclude: ## and \
                    ##not attr_name.startswith('__'):
                attrs[attr_name] = decorator(attr_value)

        return super(DecorateMethods, cls).__new__(cls, name, bases, attrs)
    
class TestClass(metaclass=DecorateMethods, decorator=wrapper):

    def __init__(self):
        print("This is my __init__ of TestClass")

    def method(self):
        print("This is my method of TestClass")
        
x = TestClass()
x.method()

TestClass is being initialized
This is my __init__ of TestClass
finished executing __init__
Decorated: method of TestClass
This is my method of TestClass
finished executing method


## with a function

In [5]:
def wrapper(function):
    @wraps(function)
    def wrapped(self, *args, **kwargs):
        if function.__name__ == '__init__':
            print(f'{self.__class__.__name__} is being initialized')
        else:
            print(f'Decorated: {function.__name__} of {self.__class__.__name__}')
        function(self, *args, **kwargs)
        print(f'finished executing {function.__name__}')
    return wrapped

def for_all_methods(decorator):
    def decorate(cls):
        for attr in cls.__dict__:
            if callable(getattr(cls, attr)):
                setattr(cls, attr, decorator(getattr(cls, attr)))
        return cls
    return decorate

@for_all_methods(wrapper)
class TestClass():

    def __init__(self):
        print("This is my __init__ of TestClass")

    def method(self):
        print("This is my method of TestClass")

x = TestClass()
x.method()

TestClass is being initialized
This is my __init__ of TestClass
finished executing __init__
Decorated: method of TestClass
This is my method of TestClass
finished executing method


In [6]:
# option to ignore methods
def for_all_methods_exclude(decorator, exclude=[]):
    def decorate(cls):
        for attr in cls.__dict__:
            if callable(getattr(cls, attr)) and attr not in exclude:
                setattr(cls, attr, decorator(getattr(cls, attr)))
        return cls
    return decorate

@for_all_methods_exclude(wrapper, ['method2'])
class TestClass():

    def __init__(self):
        print("This is my __init__ of TestClass")

    def method1(self):
        print("This is my method1 of TestClass")
        
    def method2(self):
        print("This is my method2 of TestClass")

x = TestClass()
x.method1()
x.method2()

TestClass is being initialized
This is my __init__ of TestClass
finished executing __init__
Decorated: method1 of TestClass
This is my method1 of TestClass
finished executing method1
This is my method2 of TestClass


## use a decorator inside class

In [9]:
class BaseTest:
    def wrapper(func):
        @wraps(func)
        def wrapped(self, *args, **kwargs) :
            print("start magic")
            func(self, *args, **kwargs)
            print("end magic")
        return wrapped

    @wrapper
    def bar(self) :
        print("normal call")

base_test = BaseTest()
base_test.bar()

start magic
normal call
end magic


In [8]:
class TestClass(BaseTest):

    def __init__(self):
        print("This is my __init__ of TestClass")

    @BaseTest.wrapper
    def method(self):
        print("This is my method of TestClass")
        
x = TestClass()
x.method()

This is my __init__ of TestClass
start magic
This is my method of TestClass
end magic


## subclass

In [10]:
class Logger:

    def _decorator(self, function):
        @wraps(function)
        def wrapper(*args, **kwargs):
            if function.__name__ == '__init__':
                print(f'{self.__class__.__name__} is being initialized')
            else:
                print(f'Decorated: {function.__name__} of {self.__class__.__name__}')
            output = function( *args, **kwargs)
            print(f'finished executing {function.__name__}')
            return output
        return wrapper

    def __getattribute__(self, item):
        print(item)
        value = object.__getattribute__(self, item)
        if callable(value):
            decorator = object.__getattribute__(self, '_decorator')
            return decorator(value)
        return value
    
class TestClass(Logger):

    def __init__(self):
        print("  This is my __init__ of TestClass")

    def method(self):
        print("  This is my method of TestClass")
        
x = TestClass()
x.method()

  This is my __init__ of TestClass
method
__class__
Decorated: method of TestClass
  This is my method of TestClass
finished executing method
